In [1]:
#%pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, year, month, dayofmonth, dayofweek, lower, regexp_replace, trim

spark = SparkSession.builder \
    .appName("etl_holidays_br_2026_csv") \
    .getOrCreate()

input_path = "feriados_brasil_2026.csv"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(input_path)

df = df.withColumn("date", col("date").cast("date"))

df = df.withColumn("year", year(col("date"))) \
       .withColumn("month", month(col("date"))) \
       .withColumn("day", dayofmonth(col("date"))) \
       .withColumn("day_number", dayofweek(col("date")))

df = df.withColumn(
    "day_of_week",
    when(col("day_number") == 1, "sunday")
    .when(col("day_number") == 2, "monday")
    .when(col("day_number") == 3, "tuesday")
    .when(col("day_number") == 4, "wednesday")
    .when(col("day_number") == 5, "thursday")
    .when(col("day_number") == 6, "friday")
    .when(col("day_number") == 7, "saturday")
)

df = df.withColumn(
    "weekend",
    when(col("day_number").isin(1, 7), "yes").otherwise("no")
)

df = df.withColumn("holiday", lit("yes"))

df = df.withColumn(
    "national",
    when(col("global") == True, "yes").otherwise("no")
)

df = df.withColumn(
    "regions",
    when(col("counties").isNull(), "all").otherwise(col("counties"))
)

df = df.withColumn("nome_local", col("localName"))
df = df.withColumn("english_name", col("name"))
df = df.withColumn("country_code", col("countryCode"))
df = df.withColumn("types", col("types"))

colunas_texto = [
    "day_of_week",
    "weekend",
    "holiday",
    "nome_local",
    "english_name",
    "country_code",
    "national",
    "regions",
    "types"
]

for coluna in colunas_texto:
    df = df.withColumn(coluna, lower(col(coluna)))
    df = df.withColumn(coluna, regexp_replace(col(coluna), "[\\s\\-]+", "_"))
    df = df.withColumn(coluna, regexp_replace(col(coluna), "[^a-z0-9_áàâãéèêíïóôõöúç]", ""))
    df = df.withColumn(coluna, regexp_replace(col(coluna), "_+", "_"))
    df = df.withColumn(coluna, regexp_replace(col(coluna), "^_|_$", ""))
    df = df.withColumn(coluna, trim(col(coluna)))

    df = df.withColumn(
        coluna,
        when(col(coluna).isNull() | (trim(col(coluna)) == ""), "null").otherwise(col(coluna))
    )

df_final = df.select(
    col("date"),
    col("year"),
    col("month"),
    col("day"),
    col("day_of_week"),
    col("weekend"),
    col("holiday"),
    col("nome_local"),
    col("english_name"),
    col("country_code"),
    col("national"),
    col("regions"),
    col("types")
)

df_final.show(truncate=False)

df_pandas = df_final.toPandas()
df_pandas.to_csv("etl_holidays_brazil_2026.csv", index=False, encoding="utf-8-sig")

print("etl completed successfully!")
print("file saved: holidays_brazil_2026_snake_case.csv")

+----------+----+-----+---+-----------+-------+-------+------------------------------------+------------------------------------+------------+--------+-------+-------------+
|date      |year|month|day|day_of_week|weekend|holiday|nome_local                          |english_name                        |country_code|national|regions|types        |
+----------+----+-----+---+-----------+-------+-------+------------------------------------+------------------------------------+------------+--------+-------+-------------+
|2026-01-01|2026|1    |1  |thursday   |no     |yes    |confraternização_universal          |new_years_day                       |br          |yes     |all    |public       |
|2026-02-16|2026|2    |16 |monday     |no     |yes    |carnaval                            |carnival                            |br          |yes     |all    |bank_optional|
|2026-02-17|2026|2    |17 |tuesday    |no     |yes    |carnaval                            |carnival                            |b